# Spidroin Gene Annotation

Two-step pipeline for automated spidroin annotation:

1. **Typing Agent** — parses nHMMER + miniprot results to locate spidroin loci and extract full-length sequences (start codon → stop codon).
2. **Augustus Gene Prediction** — predicts intron/exon structure and generates protein sequences for each species.

## Environment Setup
Ensure that the following dependencies have been installed (via Pixi):
```bash
pixi install
```

## Configuration

Import packages and predefine variables required throughout the notebook.

In [ ]:
from pathlib import Path

import polars as pl
from Bio import SeqIO

from spider_silkome_module import (
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
)
from spider_silkome_module.utils.run_cmd import run_cmd

In [ ]:
# Input directories
nhmmer_dir = PROCESSED_DATA_DIR / "nhmmer_search_parsed"
miniprot_dir = PROCESSED_DATA_DIR / "miniprot_output"

# Output directory (typing_results doubles as input for Augustus)
typing_output = PROCESSED_DATA_DIR / "typing_results"

# Augustus configuration
extrinsic_cfg = EXTERNAL_DATA_DIR / "extrinsic.cfg"
augustus_species = "parasteatoda"

## Step 1: Spidroin Locus Typing

Run the typing agent to pair NTD/CTD hits from nHMMER into spidroin loci, extract full-length sequences from start codon to stop codon, and write per-species results to `typing_results/`.

Outputs per species:
- `{species}.tsv` — locus table
- `{species}.gff` — locus GFF
- `spidroin_full_length.fasta` — nucleotide sequences (start → stop)
- `hints.gff` — Augustus hints

In [ ]:
run_cmd(
    f"pixi run python -m agents.typing_agent \
        --nhmmer-dir {nhmmer_dir} \
        --miniprot-dir {miniprot_dir} \
        --output {typing_output}",
    [typing_output],
)

## Step 2: Augustus Gene Prediction

Run Augustus on the spidroin sequences extracted in Step 1 to predict intron/exon structure and generate protein sequences.

Outputs per species (written to `typing_results/{species}/`):
- `augustus_output.gff` — predicted gene structures
- `augustus_output.aa` — predicted protein sequences

Per-species skip logic is handled inside `run_augustus.py`: existing output files are skipped automatically.

In [ ]:
run_cmd(
    f"pixi run python -m spider_silkome_module.run_augustus \
        --input-path {typing_output} \
        --extrinsic-cfg {extrinsic_cfg} \
        --augustus-species {augustus_species}",
    [typing_output],
    force=True,
)

## Results

Summarize the number of predicted protein sequences per species.

In [ ]:
rows = []
for sp_dir in sorted(typing_output.iterdir()):
    if not sp_dir.is_dir():
        continue
    aa_file = sp_dir / "augustus_output.aa"
    tsv_file = sp_dir / f"{sp_dir.name}.tsv"

    n_proteins = len(list(SeqIO.parse(aa_file, "fasta"))) if aa_file.exists() else None
    n_loci = None
    if tsv_file.exists():
        df_sp = pl.read_csv(tsv_file, separator="\t")
        n_loci = len(df_sp)

    rows.append({"species": sp_dir.name, "n_loci": n_loci, "n_proteins": n_proteins})

summary = pl.DataFrame(rows)
print(summary.shape)
summary